In [ ]:
import osmnx as ox
import folium

place = "SF"
tags = {"network": "Muni"}
gdf_busline = ox.features.features_from_place(place, tags)


In [ ]:
# Modify sf_boundary_full.geometry[0] to cut off anything west of -122.9
from shapely.geometry import box
from shapely.ops import clip_by_rect

# Get the original SF boundary
sf_boundary_full = ox.geocode_to_gdf("San Francisco, CA, USA")

# Create a clipping box that cuts off anything west of -122.9
# The box extends from -122.9 to the east edge, and covers the full latitude range
original_geom = sf_boundary_full.geometry[0]
bounds = original_geom.bounds  # (minx, miny, maxx, maxy)

# Create clipping rectangle: (minx, miny, maxx, maxy)
# We want to keep everything east of -122.9, so minx = -122.9
clipping_box = box(-122.9, bounds[1], bounds[2], bounds[3])

# Clip the geometry
clipped_geom = original_geom.intersection(clipping_box)

# Update the geometry in the GeoDataFrame
sf_boundary_clipped = sf_boundary_full.copy()
sf_boundary_clipped.geometry = sf_boundary_clipped.geometry.apply(
    lambda geom: geom.intersection(clipping_box) if geom.intersects(clipping_box) else None
)

# Remove any None geometries (empty intersections)
sf_boundary_clipped = sf_boundary_clipped.dropna(subset=['geometry'])

print(f"Original SF boundary bounds: minx={bounds[0]:.3f}, maxx={bounds[2]:.3f}")
print(f"Clipped SF boundary bounds: minx={sf_boundary_clipped.geometry.bounds.minx[0]:.3f}, maxx={sf_boundary_clipped.geometry.bounds.maxx[0]:.3f}")
print(f"Number of features after clipping: {len(sf_boundary_clipped)}")


In [ ]:
# Alternative approach: Get Muni bus lines using different tags
# Try getting all route relations for Muni

# Method 4: Get all routes with Muni network
tags_all_routes = {"network": "Muni", "route": ["bus", "trolleybus", "tram"]}
gdf_all_muni = ox.features.features_from_place(place, tags_all_routes)

print(f"All Muni routes: {len(gdf_all_muni)}")
if len(gdf_all_muni) > 0:
    print("Route types found:", gdf_all_muni['route'].value_counts() if 'route' in gdf_all_muni.columns else "No route column")
    print("Sample names:", gdf_all_muni['name'].head().tolist() if 'name' in gdf_all_muni.columns else "No name column")



In [ ]:
# Create map with actual bus routes
m_bus_routes = folium.Map([37.77, -122.41], zoom_start=11)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_bus_routes)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_bus_routes)

# Add bus routes as polylines
if len(gdf_all_muni) > 0:
    for idx, row in gdf_all_muni.iterrows():
        geom = row.geometry
        
        if geom.geom_type == 'LineString':
            coords = [[lat, lon] for lon, lat in geom.coords]
            name = row.get('name', f'Route {idx}')
            route_type = row.get('route', 'unknown')
            
            # Color code by route type
            color = 'blue' if route_type == 'bus' else 'green' if route_type == 'tram' else 'purple'
            
            folium.PolyLine(
                coords,
                color=color,
                weight=2,
                opacity=0.7,
                popup=f'{route_type.title()}: {name}'
            ).add_to(m_bus_routes)
            
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                coords = [[lat, lon] for lon, lat in line.coords]
                name = row.get('name', f'Route {idx}')
                route_type = row.get('route', 'unknown')
                
                color = 'blue' if route_type == 'bus' else 'green' if route_type == 'tram' else 'purple'
                
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=2,
                    opacity=0.7,
                    popup=f'{route_type.title()}: {name}'
                ).add_to(m_bus_routes)

folium.LayerControl().add_to(m_bus_routes)
m_bus_routes


In [ ]:
# Explore the gdf_all_muni data in detail
print("=== DETAILED EXPLORATION ===")
print(f"Number of features: {len(gdf_all_muni)}")
print(f"Columns: {list(gdf_all_muni.columns)}")
print(f"Geometry types: {gdf_all_muni.geometry.geom_type.value_counts()}")

print("\n=== ROUTE TYPES ===")
if 'route' in gdf_all_muni.columns:
    print("Route type distribution:")
    print(gdf_all_muni['route'].value_counts())
else:
    print("No 'route' column found")

print("\n=== NAMES ===")
if 'name' in gdf_all_muni.columns:
    print("Sample names:")
    print(gdf_all_muni['name'].head(10).tolist())
    print(f"Unique names: {gdf_all_muni['name'].nunique()}")
else:
    print("No 'name' column found")

print("\n=== REF NUMBERS ===")
if 'ref' in gdf_all_muni.columns:
    print("Sample ref numbers:")
    print(gdf_all_muni['ref'].head(10).tolist())
    print(f"Unique ref numbers: {gdf_all_muni['ref'].nunique()}")
else:
    print("No 'ref' column found")

print("\n=== SAMPLE DATA ===")
print("First few rows:")
print(gdf_all_muni.head())

print("\n=== ALL COLUMNS WITH SAMPLE VALUES ===")
for col in gdf_all_muni.columns:
    if col != 'geometry':
        print(f"\n{col}:")
        print(f"  Non-null values: {gdf_all_muni[col].notna().sum()}")
        print(f"  Sample values: {gdf_all_muni[col].dropna().head(5).tolist()}")


In [ ]:
# Filter for bus stops (bus=yes) and mark them on the map
import folium
import pandas as pd

# Filter the data for bus stops
bus_stops = gdf_all_muni[gdf_all_muni.get('bus') == 'yes']

print(f"Number of bus stops found: {len(bus_stops)}")
print(f"Bus stops columns: {list(bus_stops.columns)}")

# Create map with bus stops
m_bus_stops = folium.Map([37.77, -122.41], zoom_start=12)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_bus_stops)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_bus_stops)

# Add bus stops as markers
for idx, row in bus_stops.iterrows():
    geom = row.geometry
    
    # Get coordinates
    if geom.geom_type == 'Point':
        lat, lon = geom.y, geom.x
    elif geom.geom_type == 'MultiPoint':
        # Take the first point if it's a MultiPoint
        lat, lon = geom.geoms[0].y, geom.geoms[0].x
    else:
        # For other geometry types, get the centroid
        centroid = geom.centroid
        lat, lon = centroid.y, centroid.x
    
    # Get stop name and route info
    name = row.get('name', f'Bus Stop {idx}')
    ref = row.get('ref', '')
    display_name = f"{name} ({ref})" if ref else name
    
    # Create popup text
    popup_text = f"<b>{display_name}</b><br>"
    if 'network' in row and pd.notna(row['network']):
        popup_text += f"Network: {row['network']}<br>"
    if 'public_transport' in row and pd.notna(row['public_transport']):
        popup_text += f"Type: {row['public_transport']}<br>"
    
    # Add marker
    folium.CircleMarker(
        location=[lat, lon],
        radius=1,
        popup=popup_text,
        color='black',
        fill=True,
    ).add_to(m_bus_stops)

# Add layer control
folium.LayerControl().add_to(m_bus_stops)

m_bus_stops


In [ ]:
# Analyze all element types in the gdf data
print("=== ELEMENT TYPE ANALYSIS ===")

# Check for common element type columns
element_type_columns = ['element_type', 'type', 'highway', 'public_transport', 'amenity', 'leisure', 'tourism', 'shop', 'office']

print("Checking for element type columns:")
for col in element_type_columns:
    if col in gdf_all_muni.columns:
        print(f"✓ {col}: {gdf_all_muni[col].value_counts().head(10).to_dict()}")
    else:
        print(f"✗ {col}: Not found")

print("\n=== GEOMETRY TYPES ===")
print("Geometry type distribution:")
print(gdf_all_muni.geometry.geom_type.value_counts())

print("\n=== ALL COLUMNS WITH VALUE COUNTS ===")
for col in gdf_all_muni.columns:
    if col != 'geometry':
        unique_count = gdf_all_muni[col].nunique()
        print(f"\n{col} ({unique_count} unique values):")
        if unique_count <= 20:
            print(gdf_all_muni[col].value_counts().to_dict())
        else:
            print(f"  Sample values: {gdf_all_muni[col].dropna().head(10).tolist()}")
            print(f"  Most common: {gdf_all_muni[col].value_counts().head(5).to_dict()}")

print("\n=== SAMPLE OF EACH GEOMETRY TYPE ===")
for geom_type in gdf_all_muni.geometry.geom_type.unique():
    sample = gdf_all_muni[gdf_all_muni.geometry.geom_type == geom_type].head(3)
    print(f"\n{geom_type} examples:")
    for idx, row in sample.iterrows():
        print(f"  {idx}: {row.get('name', 'No name')} - {dict(row.drop('geometry'))}")


In [ ]:
# Get all stops for Muni 38R bus line
print("=== SEARCHING FOR MUNI 38R STOPS ===")

# Method 1: Search for 38R in various columns
print("Searching for '38R' in different columns:")

# Check if there's a route_ref or similar column
route_columns = ['route_ref', 'ref', 'name', 'route', 'network']
for col in route_columns:
    if col in gdf_all_muni.columns:
        matches = gdf_all_muni[gdf_all_muni[col].astype(str).str.contains('38R', case=False, na=False)]
        print(f"  {col}: {len(matches)} matches")
        if len(matches) > 0:
            print(f"    Sample: {matches[col].head(3).tolist()}")

# Method 2: Get all bus stops and filter for 38R
print("\n=== FILTERING BUS STOPS FOR 38R ===")
bus_stops = gdf_all_muni[gdf_all_muni.get('bus') == 'yes']
print(f"Total bus stops: {len(bus_stops)}")

# Look for 38R in bus stop names or refs
if 'name' in bus_stops.columns:
    stops_38r = bus_stops[bus_stops['name'].astype(str).str.contains('38R', case=False, na=False)]
    print(f"Stops with '38R' in name: {len(stops_38r)}")
    if len(stops_38r) > 0:
        print("38R stop names:")
        for idx, row in stops_38r.iterrows():
            print(f"  {row.get('name', 'No name')}")

if 'ref' in bus_stops.columns:
    stops_38r_ref = bus_stops[bus_stops['ref'].astype(str).str.contains('38R', case=False, na=False)]
    print(f"Stops with '38R' in ref: {len(stops_38r_ref)}")
    if len(stops_38r_ref) > 0:
        print("38R stop refs:")
        for idx, row in stops_38r_ref.iterrows():
            print(f"  {row.get('ref', 'No ref')} - {row.get('name', 'No name')}")

# Method 3: Try to get route relation for 38R
print("\n=== ATTEMPTING TO GET 38R ROUTE RELATION ===")
try:
    # Try to get the route relation for 38R
    route_38r = ox.features.features_from_place(place, {"route_ref": "38R", "network": "Muni"})
    print(f"Route relation for 38R: {len(route_38r)} features")
    if len(route_38r) > 0:
        print("38R route columns:", list(route_38r.columns))
except Exception as e:
    print(f"Error getting 38R route: {e}")

# Method 4: Search for Geary bus stops (38R runs on Geary)
print("\n=== SEARCHING FOR GEARY STREET STOPS ===")
geary_stops = bus_stops[bus_stops['name'].astype(str).str.contains('Geary', case=False, na=False)]
print(f"Stops on Geary: {len(geary_stops)}")
if len(geary_stops) > 0:
    print("Geary stops:")
    for idx, row in geary_stops.head(10).iterrows():
        print(f"  {row.get('name', 'No name')} - {row.get('ref', 'No ref')}")


In [ ]:
# Create map showing 38R stops
m_38r_stops = folium.Map([37.77, -122.41], zoom_start=12)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_38r_stops)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_38r_stops)

# Combine all potential 38R stops
all_38r_stops = []

# Add stops found by name search
if 'name' in bus_stops.columns:
    stops_38r_name = bus_stops[bus_stops['name'].astype(str).str.contains('38R', case=False, na=False)]
    all_38r_stops.append(stops_38r_name)

# Add stops found by ref search
if 'ref' in bus_stops.columns:
    stops_38r_ref = bus_stops[bus_stops['ref'].astype(str).str.contains('38R', case=False, na=False)]
    all_38r_stops.append(stops_38r_ref)

# Add Geary street stops (38R runs on Geary)
geary_stops = bus_stops[bus_stops['name'].astype(str).str.contains('Geary', case=False, na=False)]
all_38r_stops.append(geary_stops)

# Combine and remove duplicates
if all_38r_stops:
    combined_38r_stops = pd.concat(all_38r_stops).drop_duplicates()
    print(f"Total unique 38R-related stops: {len(combined_38r_stops)}")
    
    # Add markers for 38R stops
    for idx, row in combined_38r_stops.iterrows():
        geom = row.geometry
        
        # Get coordinates
        if geom.geom_type == 'Point':
            lat, lon = geom.y, geom.x
        elif geom.geom_type == 'MultiPoint':
            lat, lon = geom.geoms[0].y, geom.geoms[0].x
        else:
            centroid = geom.centroid
            lat, lon = centroid.y, centroid.x
        
        # Get stop info
        name = row.get('name', f'Stop {idx}')
        ref = row.get('ref', '')
        display_name = f"{name} ({ref})" if ref else name
        
        # Create popup
        popup_text = f"<b>38R Stop: {display_name}</b><br>"
        if 'network' in row and pd.notna(row['network']):
            popup_text += f"Network: {row['network']}<br>"
        if 'public_transport' in row and pd.notna(row['public_transport']):
            popup_text += f"Type: {row['public_transport']}<br>"
        
        # Add marker with different colors for different types
        if '38R' in str(name).upper() or '38R' in str(ref).upper():
            color = 'red'  # Direct 38R matches
            fill_color = 'red'
        elif 'GEARY' in str(name).upper():
            color = 'orange'  # Geary street stops
            fill_color = 'orange'
        else:
            color = 'blue'  # Other potential matches
            fill_color = 'lightblue'
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            popup=popup_text,
            color=color,
            fill=True,
            fillColor=fill_color,
            fillOpacity=0.8
        ).add_to(m_38r_stops)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 150px; height: 90px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>38R Stop Types:</b></p>
<p><span style="color:red">●</span> Direct 38R matches</p>
<p><span style="color:orange">●</span> Geary Street stops</p>
<p><span style="color:blue">●</span> Other matches</p>
</div>
'''
m_38r_stops.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m_38r_stops)

m_38r_stops


In [ ]:
# Connect the 38R stops to draw the bus line
m_38r_line = folium.Map([37.77, -122.41], zoom_start=12)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_38r_line)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_38r_line)

# Get all 38R-related stops
all_38r_stops = []

# Add stops found by name search
if 'name' in bus_stops.columns:
    stops_38r_name = bus_stops[bus_stops['name'].astype(str).str.contains('38R', case=False, na=False)]
    all_38r_stops.append(stops_38r_name)

# Add stops found by ref search
if 'ref' in bus_stops.columns:
    stops_38r_ref = bus_stops[bus_stops['ref'].astype(str).str.contains('38R', case=False, na=False)]
    all_38r_stops.append(stops_38r_ref)

# Add Geary street stops (38R runs on Geary)
geary_stops = bus_stops[bus_stops['name'].astype(str).str.contains('Geary', case=False, na=False)]
all_38r_stops.append(geary_stops)

# Combine and remove duplicates
if all_38r_stops:
    combined_38r_stops = pd.concat(all_38r_stops).drop_duplicates()
    print(f"Total unique 38R-related stops: {len(combined_38r_stops)}")
    
    # Extract coordinates for all stops
    stop_coords = []
    for idx, row in combined_38r_stops.iterrows():
        geom = row.geometry
        
        # Get coordinates
        if geom.geom_type == 'Point':
            lat, lon = geom.y, geom.x
        elif geom.geom_type == 'MultiPoint':
            lat, lon = geom.geoms[0].y, geom.geoms[0].x
        else:
            centroid = geom.centroid
            lat, lon = centroid.y, centroid.x
        
        stop_coords.append([lat, lon])
    
    # Sort stops by longitude (east to west) to approximate the route
    # 38R runs from downtown (east) to Richmond (west)
    stop_coords.sort(key=lambda x: x[1])  # Sort by longitude (x[1])
    
    print(f"Number of stops to connect: {len(stop_coords)}")
    
    # Draw the bus line connecting all stops
    folium.PolyLine(
        stop_coords,
        color='red',
        weight=4,
        opacity=0.8,
        popup='Muni 38R Bus Line (Connected Stops)'
    ).add_to(m_38r_line)
    
    # Add markers for each stop
    for i, (lat, lon) in enumerate(stop_coords):
        # Get stop info from the original data
        stop_info = combined_38r_stops.iloc[i]
        name = stop_info.get('name', f'Stop {i}')
        ref = stop_info.get('ref', '')
        display_name = f"{name} ({ref})" if ref else name
        
        # Create popup
        popup_text = f"<b>38R Stop {i+1}: {display_name}</b><br>"
        if 'network' in stop_info and pd.notna(stop_info['network']):
            popup_text += f"Network: {stop_info['network']}<br>"
        if 'public_transport' in stop_info and pd.notna(stop_info['public_transport']):
            popup_text += f"Type: {stop_info['public_transport']}<br>"
        
        # Color code by stop type
        if '38R' in str(name).upper() or '38R' in str(ref).upper():
            color = 'red'
            fill_color = 'red'
        elif 'GEARY' in str(name).upper():
            color = 'orange'
            fill_color = 'orange'
        else:
            color = 'blue'
            fill_color = 'lightblue'
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            popup=popup_text,
            color=color,
            fill=True,
            fillColor=fill_color,
            fillOpacity=0.8
        ).add_to(m_38r_line)
        
        # Add stop numbers
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(
                html=f'<div style="font-size: 12px; font-weight: bold; color: white; background: black; border-radius: 50%; width: 20px; height: 20px; display: flex; align-items: center; justify-content: center;">{i+1}</div>',
                icon_size=(20, 20),
                icon_anchor=(10, 10)
            )
        ).add_to(m_38r_line)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 120px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>38R Bus Line:</b></p>
<p><span style="color:red">━━━</span> Bus route line</p>
<p><span style="color:red">●</span> Direct 38R stops</p>
<p><span style="color:orange">●</span> Geary Street stops</p>
<p><span style="color:blue">●</span> Other stops</p>
<p><b>Numbers:</b> Stop sequence</p>
</div>
'''
m_38r_line.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m_38r_line)

m_38r_line


In [ ]:
# Get all Muni bus routes and their stops
print("=== GETTING ALL MUNI BUS ROUTES ===")

# Try different methods to get bus routes
methods = [
    {"route": "bus", "network": "Muni"},
    {"public_transport": "route", "network": "Muni"},
    {"operator": "San Francisco Municipal Railway"},
    {"highway": "bus_stop", "network": "Muni"},
    {"public_transport": "platform", "network": "Muni"}
]

all_routes = {}
for i, tags in enumerate(methods):
    try:
        routes = ox.features.features_from_place(place, tags)
        print(f"Method {i+1} ({tags}): {len(routes)} features")
        if len(routes) > 0:
            all_routes[f"method_{i+1}"] = routes
    except Exception as e:
        print(f"Method {i+1} failed: {e}")

# Combine all route data
if all_routes:
    # Use the method that found the most features
    best_method = max(all_routes.keys(), key=lambda k: len(all_routes[k]))
    gdf_all_routes = all_routes[best_method]
    print(f"Using {best_method} with {len(gdf_all_routes)} features")
    
    # Analyze the data
    print(f"Columns: {list(gdf_all_routes.columns)}")
    print(f"Geometry types: {gdf_all_routes.geometry.geom_type.value_counts()}")
    
    # Look for route information
    route_info_cols = ['route', 'route_ref', 'ref', 'name', 'network']
    for col in route_info_cols:
        if col in gdf_all_routes.columns:
            unique_vals = gdf_all_routes[col].nunique()
            print(f"{col}: {unique_vals} unique values")
            if unique_vals <= 20:
                print(f"  Values: {gdf_all_routes[col].value_counts().to_dict()}")
            else:
                print(f"  Sample: {gdf_all_routes[col].dropna().head(5).tolist()}")
else:
    print("No route data found")
    gdf_all_routes = gdf_all_muni  # Fallback to original data


In [ ]:
# Create map with all Muni bus routes
m_all_routes = folium.Map([37.77, -122.41], zoom_start=11)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_all_routes)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_all_routes)

# Get all bus stops
bus_stops = gdf_all_routes[gdf_all_routes.get('bus') == 'yes']
print(f"Total bus stops found: {len(bus_stops)}")

# Group stops by route if possible
if 'route_ref' in bus_stops.columns:
    # Group by route_ref
    route_groups = bus_stops.groupby('route_ref')
    print(f"Found {len(route_groups)} route groups")
elif 'ref' in bus_stops.columns:
    # Group by ref
    route_groups = bus_stops.groupby('ref')
    print(f"Found {len(route_groups)} route groups")
else:
    # Group by name patterns (look for route numbers)
    bus_stops['route_pattern'] = bus_stops['name'].astype(str).str.extract(r'(\d+[A-Z]?)', expand=False)
    route_groups = bus_stops[bus_stops['route_pattern'].notna()].groupby('route_pattern')
    print(f"Found {len(route_groups)} route groups by pattern")

# Color palette for different routes
import matplotlib.pyplot as plt
import numpy as np

# Generate colors for routes
n_routes = len(route_groups)
colors = plt.cm.tab20(np.linspace(0, 1, min(n_routes, 20)))
if n_routes > 20:
    colors = plt.cm.tab20(np.linspace(0, 1, 20))
    colors = np.tile(colors, (n_routes // 20 + 1, 1))[:n_routes]

route_colors = {}
for i, (route_name, group) in enumerate(route_groups):
    if i < len(colors):
        color = colors[i]
        route_colors[route_name] = f"#{int(color[0]*255):02x}{int(color[1]*255):02x}{int(color[2]*255):02x}"
    else:
        route_colors[route_name] = "#000000"  # Black for overflow

print(f"Processing {len(route_groups)} routes...")

# Process each route
route_count = 0
for route_name, group in route_groups:
    if len(group) < 2:  # Skip routes with less than 2 stops
        continue
        
    route_count += 1
    if route_count > 50:  # Limit to first 50 routes for performance
        break
        
    # Extract coordinates
    stop_coords = []
    for idx, row in group.iterrows():
        geom = row.geometry
        
        if geom.geom_type == 'Point':
            lat, lon = geom.y, geom.x
        elif geom.geom_type == 'MultiPoint':
            lat, lon = geom.geoms[0].y, geom.geoms[0].x
        else:
            centroid = geom.centroid
            lat, lon = centroid.y, centroid.x
        
        stop_coords.append([lat, lon])
    
    if len(stop_coords) < 2:
        continue
    
    # Sort stops by longitude for route direction
    stop_coords.sort(key=lambda x: x[1])
    
    # Draw the route line
    folium.PolyLine(
        stop_coords,
        color=route_colors.get(route_name, '#000000'),
        weight=5,
        opacity=0.8,
        popup=f'Route {route_name} ({len(stop_coords)} stops)'
    ).add_to(m_all_routes)
    
    # Add stop markers (smaller for performance)
    for i, (lat, lon) in enumerate(stop_coords):
        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color=route_colors.get(route_name, '#000000'),
            fill=True,
            fillColor=route_colors.get(route_name, '#000000'),
            fillOpacity=0.7,
            popup=f'Route {route_name} - Stop {i+1}'
        ).add_to(m_all_routes)

print(f"Added {route_count} routes to the map")

# Add legend
legend_html = f'''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 150px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>Muni Bus Routes:</b></p>
<p>Total routes: {route_count}</p>
<p>Each route has a unique color</p>
<p>Lines show route paths</p>
<p>Dots show individual stops</p>
</div>
'''
m_all_routes.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m_all_routes)

m_all_routes


In [ ]:
# Get golf courses, aquariums, zoos, and museums in SF
print("=== GETTING AMENITIES IN SF ===")

# Define amenity types to search for
amenity_types = {
    'golf_course': 'Golf Courses',
    'aquarium': 'Aquariums', 
    'zoo': 'Zoos',
    'museum': 'Museums'
}

# Get amenities for each type
amenities_data = {}
for amenity_type, display_name in amenity_types.items():
    try:
        # Search for amenities within SF boundary
        amenities = ox.features.features_from_place(place, {"amenity": amenity_type})
        print(f"{display_name}: {len(amenities)} found")
        
        if len(amenities) > 0:
            amenities_data[amenity_type] = {
                'data': amenities,
                'display_name': display_name,
                'color': {
                    'golf_course': 'green',
                    'aquarium': 'blue', 
                    'zoo': 'orange',
                    'museum': 'purple'
                }[amenity_type]
            }
    except Exception as e:
        print(f"Error getting {display_name}: {e}")

print(f"Total amenity types found: {len(amenities_data)}")

# Also try to get additional leisure facilities
leisure_types = {
    'golf_course': 'Golf Courses (Leisure)',
    'zoo': 'Zoos (Leisure)',
    'museum': 'Museums (Leisure)'
}

for leisure_type, display_name in leisure_types.items():
    try:
        leisure_facilities = ox.features.features_from_place(place, {"leisure": leisure_type})
        print(f"{display_name}: {len(leisure_facilities)} found")
        
        if len(leisure_facilities) > 0:
            # Add to existing data or create new
            if leisure_type in amenities_data:
                # Combine with existing data
                combined = pd.concat([amenities_data[leisure_type]['data'], leisure_facilities]).drop_duplicates()
                amenities_data[leisure_type]['data'] = combined
                print(f"  Combined total for {leisure_type}: {len(combined)}")
            else:
                amenities_data[leisure_type] = {
                    'data': leisure_facilities,
                    'display_name': display_name,
                    'color': {
                        'golf_course': 'green',
                        'zoo': 'orange',
                        'museum': 'purple'
                    }[leisure_type]
                }
    except Exception as e:
        print(f"Error getting {display_name}: {e}")

# Show sample data for each type
for amenity_type, info in amenities_data.items():
    print(f"\n{info['display_name']} sample data:")
    sample = info['data'].head(3)
    for idx, row in sample.iterrows():
        name = row.get('name', 'No name')
        print(f"  {name}")


In [ ]:
# Create map with amenities as toggleable layers
m_amenities = folium.Map([37.77, -122.41], zoom_start=11)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_amenities)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_amenities)

# Add amenities as separate layers
for amenity_type, info in amenities_data.items():
    # Create a feature group for this amenity type
    amenity_group = folium.FeatureGroup(name=info['display_name'])
    
    # Add markers for each amenity
    for idx, row in info['data'].iterrows():
        geom = row.geometry
        
        # Get coordinates
        if geom.geom_type == 'Point':
            lat, lon = geom.y, geom.x
        elif geom.geom_type == 'MultiPoint':
            lat, lon = geom.geoms[0].y, geom.geoms[0].x
        else:
            centroid = geom.centroid
            lat, lon = centroid.y, centroid.x
        
        # Get amenity info
        name = row.get('name', f'{amenity_type.title()} {idx}')
        amenity_type_actual = row.get('amenity', row.get('leisure', amenity_type))
        
        # Create popup
        popup_text = f"<b>{name}</b><br>"
        popup_text += f"Type: {amenity_type_actual.title()}<br>"
        
        # Add additional info if available
        if 'website' in row and pd.notna(row['website']):
            popup_text += f"Website: <a href='{row['website']}' target='_blank'>{row['website']}</a><br>"
        if 'phone' in row and pd.notna(row['phone']):
            popup_text += f"Phone: {row['phone']}<br>"
        if 'addr:street' in row and pd.notna(row['addr:street']):
            popup_text += f"Address: {row['addr:street']}"
            if 'addr:city' in row and pd.notna(row['addr:city']):
                popup_text += f", {row['addr:city']}"
            popup_text += "<br>"
        
        # Choose icon based on amenity type
        if amenity_type == 'golf_course':
            icon = 'fa-golf-ball-tee'
        elif amenity_type == 'aquarium':
            icon = 'fa-fish'
        elif amenity_type == 'zoo':
            icon = 'fa-paw'
        elif amenity_type == 'museum':
            icon = 'fa-building-columns'
        else:
            icon = 'fa-map-marker'
        
        # Add marker with custom icon
        folium.Marker(
            location=[lat, lon],
            popup=popup_text,
            icon=folium.Icon(
                color=info['color'],
                icon=icon,
                prefix='fa'
            )
        ).add_to(amenity_group)
    
    # Add the feature group to the map
    amenity_group.add_to(m_amenities)

# Add layer control
folium.LayerControl().add_to(m_amenities)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 200px; height: 150px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>Amenities Legend:</b></p>
<p><i class="fa fa-golf-ball-tee" style="color:green"></i> Golf Courses</p>
<p><i class="fa fa-fish" style="color:blue"></i> Aquariums</p>
<p><i class="fa fa-paw" style="color:orange"></i> Zoos</p>
<p><i class="fa fa-building-columns" style="color:purple"></i> Museums</p>
<p><small>Use layer control to toggle visibility</small></p>
</div>
'''
m_amenities.get_root().html.add_child(folium.Element(legend_html))

m_amenities


In [ ]:
# Draw a 1-mile circle centered on Geary and Masonic
import folium
from folium import Circle
import math

# Geary and Masonic intersection coordinates
# This is approximately 37.7849°N, 122.4594°W
center_lat = 37.7849
center_lon = -122.4594

# Convert 1 mile to meters (1 mile = 1609.34 meters)
radius_meters = 1609.34

# Create map with the circle
m_circle = folium.Map([center_lat, center_lon], zoom_start=14)

# Add the clipped SF boundary
folium.GeoJson(
    sf_boundary_clipped.to_json(),
    style_function=lambda feature: {
        'fillColor': 'lightblue',
        'color': 'red',
        'weight': 3,
        'opacity': 0.8,
        'fillOpacity': 0.3
    },
    name="SF Boundary (Clipped)"
).add_to(m_circle)

# Add vertical line at -122.9° West longitude
longitude_line = [
    [37.6, -122.9],
    [37.9, -122.9]
]

folium.PolyLine(
    longitude_line,
    color='orange',
    weight=4,
    opacity=0.8,
    popup='-122.9° West Longitude'
).add_to(m_circle)

# Add the 1-mile circle
folium.Circle(
    location=[center_lat, center_lon],
    radius=radius_meters,
    popup='1 Mile Radius from Geary & Masonic',
    color='red',
    weight=3,
    fill=True,
    fillColor='red',
    fillOpacity=0.2
).add_to(m_circle)

# Add a marker at the center point
folium.Marker(
    location=[center_lat, center_lon],
    popup='Geary & Masonic Intersection',
    icon=folium.Icon(color='red', icon='fa-crosshairs', prefix='fa')
).add_to(m_circle)

# Add amenities within the circle
if 'amenities_data' in locals():
    for amenity_type, info in amenities_data.items():
        # Create a feature group for this amenity type
        amenity_group = folium.FeatureGroup(name=f"{info['display_name']} (1-mile radius)")
        
        # Add markers for amenities within the circle
        for idx, row in info['data'].iterrows():
            geom = row.geometry
            
            # Get coordinates
            if geom.geom_type == 'Point':
                lat, lon = geom.y, geom.x
            elif geom.geom_type == 'MultiPoint':
                lat, lon = geom.geoms[0].y, geom.geoms[0].x
            else:
                centroid = geom.centroid
                lat, lon = centroid.y, centroid.x
            
            # Check if the amenity is within 1 mile of the center
            # Calculate distance using Haversine formula
            def haversine_distance(lat1, lon1, lat2, lon2):
                R = 6371000  # Earth's radius in meters
                dlat = math.radians(lat2 - lat1)
                dlon = math.radians(lon2 - lon1)
                a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
                c = 2 * math.asin(math.sqrt(a))
                return R * c
            
            distance = haversine_distance(center_lat, center_lon, lat, lon)
            
            if distance <= radius_meters:
                # Get amenity info
                name = row.get('name', f'{amenity_type.title()} {idx}')
                amenity_type_actual = row.get('amenity', row.get('leisure', amenity_type))
                
                # Create popup
                popup_text = f"<b>{name}</b><br>"
                popup_text += f"Type: {amenity_type_actual.title()}<br>"
                popup_text += f"Distance: {distance/1609.34:.2f} miles<br>"
                
                # Add additional info if available
                if 'website' in row and pd.notna(row['website']):
                    popup_text += f"Website: <a href='{row['website']}' target='_blank'>{row['website']}</a><br>"
                if 'phone' in row and pd.notna(row['phone']):
                    popup_text += f"Phone: {row['phone']}<br>"
                if 'addr:street' in row and pd.notna(row['addr:street']):
                    popup_text += f"Address: {row['addr:street']}"
                    if 'addr:city' in row and pd.notna(row['addr:city']):
                        popup_text += f", {row['addr:city']}"
                    popup_text += "<br>"
                
                # Choose icon based on amenity type
                if amenity_type == 'golf_course':
                    icon = 'fa-golf-ball-tee'
                elif amenity_type == 'aquarium':
                    icon = 'fa-fish'
                elif amenity_type == 'zoo':
                    icon = 'fa-paw'
                elif amenity_type == 'museum':
                    icon = 'fa-building-columns'
                else:
                    icon = 'fa-map-marker'
                
                # Add marker with custom icon
                folium.Marker(
                    location=[lat, lon],
                    popup=popup_text,
                    icon=folium.Icon(
                        color=info['color'],
                        icon=icon,
                        prefix='fa'
                    )
                ).add_to(amenity_group)
        
        # Add the feature group to the map
        amenity_group.add_to(m_circle)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 250px; height: 200px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>1-Mile Circle from Geary & Masonic:</b></p>
<p><span style="color:red">●</span> Center point (Geary & Masonic)</p>
<p><span style="color:red">○</span> 1-mile radius circle</p>
<p><i class="fa fa-golf-ball-tee" style="color:green"></i> Golf Courses</p>
<p><i class="fa fa-fish" style="color:blue"></i> Aquariums</p>
<p><i class="fa fa-paw" style="color:orange"></i> Zoos</p>
<p><i class="fa fa-building-columns" style="color:purple"></i> Museums</p>
<p><small>Only amenities within 1 mile are shown</small></p>
</div>
'''
m_circle.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m_circle)

m_circle
